In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

transactions_data = [
    (1, 101, 100.0, "2025-01-01"),
    (2, 101, 200.0, "2025-01-02"),
    (3, 101, 150.0, "2025-01-03"),
    (4, 102, 300.0, "2025-01-01"),
    (5, 102, 400.0, "2025-01-02"),
    (6, 103, 250.0, "2025-01-01"),
    (7, 103, 250.0, "2025-01-02"),
    (8, 103, 500.0, "2025-01-03")
]

transactions_df = spark.createDataFrame(
    transactions_data,
    ["transaction_id", "customer_id", "amount", "transaction_date"]
)

In [0]:
# Part A — Ranking
window_spec = Window.partitionBy("customer_id").orderBy("transaction_date")

result_row_number = transactions_df.withColumn(
    "row_num", F.row_number().over(window_spec)
)

result_rank = transactions_df.withColumn("row_rank", F.rank().over(window_spec))

result_dense_rank = transactions_df.withColumn(
    "row_dense_rank", F.dense_rank().over(window_spec)
)

# Part B — Historical Analysis
result_lag = transactions_df.withColumn(
    "previous_amount", F.lag("amount", 1).over(window_spec)
)

result_lead = transactions_df.withColumn(
    "next_amount", F.lead("amount", 1).over(window_spec)
)

result_difference_between_previous = transactions_df.withColumn(
    "previous_amount", F.lag("amount", 1).over(window_spec)
).withColumn("difference_amount", F.col("amount") - F.col("previous_amount"))

# Part C — Aggregation
result_total_amount_per_customer = (
    transactions_df
    .withColumn(
        "total_amount",
        F.sum("amount").over(
            Window
            .partitionBy("customer_id")
        )
    )
)

result_running_total_each_customer = (
    transactions_df
    .withColumn(
        "running_total",
        F.sum("amount").over(window_spec)
    )
)

# Part D — Deduplication
customer_duplicates_data = [
    (101, "Ali", "2025-01-01 10:00:00"),
    (101, "Ali Ahmed", "2025-01-03 14:30:00"),
    (101, "Ali A. Khan", "2025-01-05 09:15:00"),
    (102, "John", "2025-01-01 11:00:00"),
    (102, "John Smith", "2025-01-04 16:45:00"),
    (103, "Sara", "2025-01-02 08:30:00")
]

customer_columns = ["customer_id", "customer_name", "updated_at"]

customer_duplicates_df = spark.createDataFrame(customer_duplicates_data, schema=customer_columns)

latest_record = (
    customer_duplicates_df
    .withColumn(
        "row_num",
        F.row_number().over(
            Window
            .partitionBy("customer_id")
            .orderBy(F.desc("updated_at"))
        )
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

customer_duplicates_df.orderBy(F.col("updated_at").desc()).display()
latest_record.display()